# Chapter 22 - Monte Carlo Variance Reduction and Scalar Flux Estimation

So far we've been doing a process called *analog* Monte Carlo. We can change the way the Monte Carlo "game" is played and adjust our sampling to be more favorable. 

## 22.1 Implicit Capture and Particle Weights

*Implicit Capture* is a method where we modify our sampling in a problem where capture is important. We modify our sampling such that particle weights are adjusted as a particle transits the problem. We don't sample every capture reaction, but instead reduce a particle weight to account for capture reactions. 

The algorithm for implicit capture is:
1. Create a counter, $t = 0$ to track the number of neutrons that get through.  
2. Create neutron with $\mu$ sampled from the uniform distribution $\mu \in [0,1]$. Set $x = 0$. Set the particle’s weight to be $w = 1/N$.  
3. Sample randomly a distance to scatter, $l$, from the exponential distribution.  
4. Move the particle to $x = x + l\mu$.  
5. Reduce the weight of the particle by a factor $\exp(-\Sigma_\gamma s)$.  
6. Check to see if $x > 3$. If so $t = t + w$, and go to 2. Otherwise, if $x < 0$ go to step 2.  
7. Go back to step 3.  


A couple of notes on this new algorithm. Now each particle has a weight and that is what we
sum up to get the fraction of neutrons that leak out per unit time. Also, when we sample a
distance to collision, we only sample a distance to scatter.

One drawback of implicit capture is that it can result in the tracking of particles that have
a very small weight. After traveling a large distance, the weight could be much less than
the initial weight and the particle could contribute only a small amount to the result. This
is wasted computational effort tracking these low weight particles; it would be better to stop
tracking them. For this purpose we introduce a cutoff weight. After decreasing from the initial
weight by more than the cutoff, we treat the particle using analog tracking so that it can die
via absorption.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [14]:
def slab_transmission(Sig_s,Sig_a,thickness,N,isotropic=False, implicit_capture = True, cutoff = 1.0e-3):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
    Sig_s:     The scattering macroscopic x-section
    Sig_a:     The absorption macroscopic x-section
    thickness: Width of the slab
    N:         Number of neutrons to simulate
    isotropic: Are the neutrons isotropic or a beam
    
    Returns:
    transmission:  The fraction of neutrons that made it through
    """
    imp_input = implicit_capture
    Sig_t = Sig_a + Sig_s
    iSig_t = 1.0/Sig_t
    iSig_s = 1.0/(Sig_s + 1.0e-14)
    
    transmission = 0.0
    N = int(N)
    initial_weight = 1.0/N

    for i in range(N):
        if (isotropic):
            mu = np.random.random()
        else:
            mu = 1.0
        
        x = 0
        alive = 1
        weight = initial_weight
       
        while (alive):

            if (weight < cutoff*initial_weight):
                implicit_capture = False

            if (implicit_capture):
                #get distance to collision
                l = -np.log(1-np.random.random())*iSig_s 
            else:
                #get distance to collision
                l = -np.log(1-np.random.random())*iSig_t

            #make sure that l is not too large. If it is, move it to the edge.
            if (mu > 0):
                val = (thickness-x)/mu
                l = np.min([l,val])
            else:
                l = np.min([l,-x/mu])
            
            #move particle
            x += l*mu
            if (implicit_capture):
                weight *= np.exp(-l*Sig_a)
            
            #still in the slab? 
            if (np.abs(x-thickness) < 1.0e-14):
                transmission += weight
                alive = 0
            elif (x<= 1.0e-14):
                alive = 0
            else:
                if (implicit_capture):
                    mu = np.random.uniform(-1,1)
                else:
                    #scatter or absorb
                    if (np.random.random() < Sig_s/Sig_t): 
                        #scatter, pick new mu
                        mu = np.random.uniform(-1,1)
                    else: #absorbed
                        alive = 0
    return transmission



Now let's try a problem where there's no scattering and only absorption. Implicit capture works well in problems where capture is important; a problem with no scattering is an extreme case of this. 

With a pure absorber, we can calculate the exact answer for transmission. Let's see the difference with implicit capture and analog Monte Carlo using one particle in the simulation. 

In [15]:
N = 1
Sigma_s = 0.0
Sigma_a = 2.0
thickness = 3
transmission = slab_transmission(Sigma_s,Sigma_a, thickness,
                                 N, isotropic=False, implicit_capture=True)
print("The fraction that made it through using implicit capture was", transmission, "with a percent error of",
      np.abs(transmission - np.exp(-6))/np.exp(-6)*100,"%")
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, 
                                 N, isotropic=False, implicit_capture=False)
print("The fraction that made it through using analog tracking was", transmission, "with a percent error of", 
      np.abs(transmission - np.exp(-6))/np.exp(-6)*100,"%")

The fraction that made it through using implicit capture was 0.0024787521766663585 with a percent error of 0.0 %
The fraction that made it through using analog tracking was 0.0 with a percent error of 100.0 %


With an isotropic particle distribution in the same problem, we won't get the exact answer with a single particle, but we can expect it to perform better in a problem with the same material properties and dimensions. Let's try with 10,000 particles and compare them... 

In [16]:
true_sol = 0.00031825746369040646727
N=10000
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, N, isotropic=True, implicit_capture=True)
print("The fraction that made it through using implicit capture was", transmission, "with a percent error of",np.abs(transmission - true_sol)/true_sol*100,"%")
transmission = slab_transmission(Sigma_s,Sigma_a, thickness,
                                 N, isotropic=True, implicit_capture=False)
print("The fraction that made it through using analog tracking was",
      transmission, "with a percent error of",
      np.abs(transmission - true_sol)/true_sol*100,"%")

The fraction that made it through using implicit capture was 0.0003251902550934969 with a percent error of 2.1783594083545186 %
The fraction that made it through using analog tracking was 0.00030000000000000003 with a percent error of 5.736696157475466 %


Variance reduction methods, like implicit capture, reduce the time or number of particles needed to get to a particular solution (with a desired uncertainty) by changing the "game" that is being played to simulate these particles. As we discussed in implicit capture, because we changed how we sample distance to collision, we had to modify the particle weight accordingly and consistently. 


## Other variance reduction methods

Many variance reduction methods exist. Two additional methods are russian roulette and splitting. In russian roulette, we systematically get rid of particles and increase surviving particle weights so that the total weight of the problem remains unchanged. 

In splitting we increase the number of particles and decrease their weights to the total weight of the problem remains unchanged. 


<div class="alert alert-block alert-info">
<h2> <b>Think:</b> When would you want to use rouletting? When would you want to use splitting?</h2>
</div>

In [ ]:

def russian_roulette(weight, wa):
    """Perform Russian roulette on neutron"""

    pk = 1 - weight/wa        # probability of killing particle

    if np.random.random() < pk:
        alive = 0
        weight_final = 0
    else:
        alive = 1
        weight_final = wa

    return alive, weight_final

In [ ]:
def split(neutron, wd):
    """MCNP-style stochastic splitting"""

    import numpy as np

    w = neutron[0]
    n = w / wd

    m = int(np.floor(n))
    f = n - m

    if np.random.random() < f:
        num_split = m + 1
    else:
        num_split = m

    num_split = max(1, num_split)

    wsplit = w / num_split

    new_neutrons = np.zeros((num_split-1, len(neutron)))

    neutron[0] = wsplit

    for i in range(num_split-1):
        new_neutrons[i,:] = neutron.copy()

    return new_neutrons

## 22.1.1 A Figure of Merit

A way of measuring the benefit of a variance reduction technique is through the quantity
called the **figure of merit (FOM)**

$$
\text{FOM} = \frac{1}{\sigma^2 T}.
$$

It should be said that this is *a* figure of merit, not *the* figure of merit, because other
definitions are possible. In the expression for FOM, $\sigma^2$ is the variance in an estimate
and $T$ is the time it takes to obtain that estimate. The benefit of the FOM is that it gives
us a way to determine whether the cost of a variance reduction technique is justified by any
increase in computational time.

We can test this using our slab problem. We will run the problem 20 times for each number
of particles, compute the time required for each run, and calculate the variance of the results.
From these values we can compute and plot the FOM.

Remember that **a larger FOM is better**, since it corresponds to lower variance and/or
shorter computation time. We expect that the implicit capture method should perform
better because it appears to give smaller error (see above), and the exponential evaluations
required by implicit capture should not be computationally expensive compared with the
cost of tracking particles.

In [ ]:
import numpy as np
import time

N_parts = [10,100,1000,2000,4000,8000,16000,32000,64000,128000]
Ntimes = 20

solution_implicit = np.zeros((len(N_parts), Ntimes))
solution_analog   = np.zeros((len(N_parts), Ntimes))

times_implicit = np.zeros(len(N_parts))
times_analog   = np.zeros(len(N_parts))

var_implicit = np.zeros(len(N_parts))
var_analog   = np.zeros(len(N_parts))

Sigma_s = 0.75
Sigma_a = 2.0 - Sigma_s
thickness = 3

count = 0

for N in N_parts:
    
    for replicate in range(Ntimes):

        tmp = time.perf_counter()
        solution_implicit[count, replicate] = slab_transmission(
            Sigma_s, Sigma_a, thickness, N,
            isotropic=True,
            implicit_capture=True,
            cutoff=1e-2
        )
        times_implicit[count] += (time.perf_counter() - tmp) / Ntimes

        tmp = time.perf_counter()
        solution_analog[count, replicate] = slab_transmission(
            Sigma_s, Sigma_a, thickness, N,
            isotropic=True,
            implicit_capture=False
        )
        times_analog[count] += (time.perf_counter() - tmp) / Ntimes

    var_implicit[count] = np.std(solution_implicit[count, :])**2
    var_analog[count]   = np.std(solution_analog[count, :])**2

    count += 1

<img src="Uploaded Media/22.1.png" width="400">

In this example, we see that the FOM for implicit capture is about an order of magnitude
larger than analog tracking. This means that implicit capture can get the same variance as
analog tracking in one-tenth the time.

## 22.2 Estimating Scalar Flux

So far we've been focusing on how to calculate transmissions through slabs, but we may want to consider other behaviors in our problems. 

Our transmission calculations so far have involved particles crossing a surface, but let's consider how we might quantify material happening within a volume, or cell. The neutron flux can be estimated in two ways: through collision estimators and track length estimators.

### 22.2.1 Collision Esimators

Consider the reaction rate in a volume, defined by

$$
R = \int_V dV \, \Sigma_t(\mathbf{r}) \, \phi(\mathbf{r}).
$$

If inside the region the cross-section is constant, we can compute the average scalar flux via
the relation

$$
\bar{\phi} = \frac{1}{V}\int_V dV \, \phi(\mathbf{r}) = \frac{R}{V\Sigma_t}.
$$

Therefore, if we sum (or **tally**) the weight from each collision inside the volume and divide
that count by the total cross-section, we obtain an estimate of the scalar flux.

Notice, however, that we cannot use this in voids because $\Sigma_t = 0$. In that case the
collision estimator cannot be applied. 

For **implicit capture**, we can compute the scattering rate and divide by $\Sigma_s$.


The slab problem from above will be modified for this purpose. We will introduce a mesh
onto the problem and count the reactions in each mesh cell. Also, instead of a source on
the boundary, we will add a **volumetric source** to the problem.

The source will be uniform between $a$ and $b$. Therefore, we must sample both the
position of the neutron’s birth and its direction. The birth position will be sampled from
the interval $[a,b]$, and the direction cosine will be sampled from

$$
\mu \in [-1,1].
$$

### 22.2.2 Track-length Estimator

Another type of estimator for the scalar flux uses the definition of the scalar flux to
estimate it. Recall that the scalar flux is the **rate-density at which neutrons generate
track length**. Therefore, for a given cell, every time a neutron moves inside it we sum the
weight of the neutron times its path length in the cell. We then divide this by the volume
of the region.

We can write this in equation form as

$$
\bar{\phi} = \frac{1}{V}\sum_{\text{neutrons}} (\text{weight} \times \text{path length}).
$$

For implicit capture, the weight changes continuously while the neutron moves through a
region. Therefore, we integrate the weight along the particle track to determine its
contribution to the estimator.

A neutron traveling a distance $s$ inside a region contributes

$$
\text{contribution}
=
\int_{0}^{s} ds' \, w_0 e^{-\Sigma_a s'}
=
\frac{w_0}{\Sigma_a}\left(1 - e^{-\Sigma_a s}\right),
$$

where $w_0$ is the particle weight at the beginning of the track segment.

To implement this estimator we will reformulate how we do our tracking. We will make
our method work by checking the distance to collision against the distance to the edge of
a cell. Whichever distance is shorter, that event occurs: either the neutron has a collision or
it moves to the next cell and a new distance to collision is sampled. This means that the
neutrons will step through the problem cell by cell. This will slow down the code, because
now the particles can only take steps limited by the width of the mesh cells.




## 22.3 Stratified Sampling

The idea behind **stratified sampling** is to control the randomness in the simulation. We
want to use random numbers to simulate neutron interactions, but there is no guarantee that
purely random samples will not cluster together. Stratified sampling is a way to spread out
the samples more evenly.

It is easiest to think about stratification in terms of a single random variable uniformly
distributed between 0 and 1. There are several possible formulations, but the most
straightforward approach divides the interval $[0,1]$ into $S$ bins (or **strata**) of equal size.
We then pick a bin by generating a random integer between $0$ and $S-1$. Inside that bin
we randomly select a location.

If we perform this sampling so that **each bin receives the same number of samples**, we
expect the samples to fill the interval $[0,1]$ more evenly than with simple random sampling.

For this example, the code to produce the stratified samples is straightforward.


In [ ]:
def strat_sample(N,S):    
    """Create N samples in S strata.
    N must be divisible by S
    Inputs:
    N:             number of samples
    S:             number of strata
    Returns:
    place_in_bin:  a numpy vector containing the samples
    """
    N = N + (N % S)
    assert(N%S == 0 )
    dS = 1.0/S
    bins = np.zeros(N,dtype=int)
    count = 0
    for i in range(N//S):
        bins[count:count+S] = np.random.permutation(S)
        count += S
    place_in_bin = np.random.uniform(-0.5*dS,0.5*dS,N) + (bins+0.5)*dS
    return place_in_bin

This can be extended into two dimensions (SxS strata). 

The following code gives a stratification in two-dimensions. It will increase the number of
samples to match the desired number of strata, if needed. It also allows the number of strata in each dimension to differ.

In [ ]:
import numpy as np
import random

def strat_sample_2D(N, S1, S2):
    """Create N samples in S1*S2 strata.

    Inputs:
        N  : number of samples
        S1 : number of strata in dimension 1
        S2 : number of strata in dimension 2

    Returns:
        samples : N by 2 numpy array containing the samples
    """

    # number of bins
    bins = S1 * S2

    # make sure we have enough points
    if N < bins:
        N = bins

    N -= (N % bins)
    Num_per_bin = N // bins
    assert (N % bins == 0)

    samples = np.zeros((N, 2))
    count = 0

    for bin_x in range(S1):
        for bin_y in range(S2):
            for i in range(Num_per_bin):

                center = (bin_x/S1 + 0.5/S1, bin_y/S2 + 0.5/S2)

                samples[count, 0] = center[0] + random.uniform(-0.5, 0.5) / S1
                samples[count, 1] = center[1] + random.uniform(-0.5, 0.5) / S2

                count += 1

    return samples

Sampling a 2-D space with 5, 10, and 50 strata in each dimension (for a total of

$$
5^2 = 25,\qquad 10^2 = 100,\qquad \text{and} \qquad 50^2 = 2500
$$

strata), all with 2500 samples, are compared with unstratified sampling in the following figure.

<img src="Uploaded Media/22.3.png" width="400">


We can apply 2-D stratification by using it to pick the initial position and μ for the source
particles in the slab. This should decrease the variance in our calculation when we increase
the number of neutrons sampled to have a large number of strata. 

## 22.4 Complete Monte Carlo Code

Bringing this all together, below is a slab function that can use implicit capture, tallies the flux with both collision and track length estimators, and with stratified sampling: 

In [ ]:
def slab_source(Nx,Sig_s,Sig_a,thickness,a,b,N,Q,
                implicit_capture = True,
                cutoff = 1.0e-3,
                stratified = [1,1]):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
        Nx: The number of grid points
        Sig_s: The scattering macroscopic x-section
        Sig_a: The absorption macroscopic x-section
        thickness: Width of the slab
        a,b: Endpoints of Source
        N: Number of neutrons to simulate
        implicit_capture: Do we run implicit capture
        cutoff: At what level do we stop implicit capture
        stratified: Use stratified sampling in space and angle
            Specify a list of length two with the number of
            strata in each dimension; default [1,1] for unstratified
    Returns:
        scalar_flux: scalar_flux_tl: transmission: The fraction of neutrons that made it through
            The scalar flux in each of the Nx cells
            The scalar flux in each of the Nx cells
            from track length estimator
        X:
            The value of the cell centers in the mesh
    """
    imp_input = implicit_capture
    dx = thickness/Nx
    X = np.linspace(dx*0.5, thickness - 0.5*dx,Nx)
    scalar_flux = np.zeros(Nx)
    scalar_flux_tl = np.zeros(Nx)
    assert (Sig_s.size == Nx) and (Sig_a.size == Nx)
    Sig_t = Sig_a + Sig_s
    iSig_t = 1.0/Sig_t
    iSig_s = 1.0/(Sig_s+1.0e-14)
    iSig_a = 1.0/(Sig_a+1.0e-14)
    leak_left = 0.0
    leak_right = 0
    N = int(N)

    #make a vector of the initial positions and mus
    samples = strat_sample_2D(N,stratified[0],stratified[1])
    xs = samples[:,0]*(b-a) + a #adjust to bounds of source
    mus = (samples[:,1]-0.5)*2 #shift to range -1 to 1
    N = int(xs.size)

    #the initial weight does not change
    init_weight = Q*thickness/N

    for i in range(N):
        mu = mus[i]
        x = xs[i]
        alive = 1
        weight = init_weight

        #which cell am I in
        cell = int(x/dx)
        implicit_capture = imp_input

        while (alive):
            if (weight < cutoff*init_weight):
                implicit_capture = False

            if (implicit_capture):
                l = -np.log(1-random.random())*iSig_s[cell]
            else:
                #get distance to collision
                l = -np.log(1-random.random())*iSig_t[cell]

            #compare distance to collision to distance to cell edge
            distance_to_edge = ((mu > 0.0)*( (cell+1)*dx - x) +
                                (mu<0.0)*( x - cell*dx) + 1.0e-8)/np.fabs(mu)

            if (distance_to_edge < l):
                l = distance_to_edge
                collide = 0
            else:
                collide = 1

            x += l*mu #move particle

            #score track length tally
            if (implicit_capture):
                scalar_flux_tl[cell] += weight*(1.0 -
                                                np.exp(-l*Sig_a[cell]))*iSig_a[cell]
            else:
                scalar_flux_tl[cell] += weight*l

            if (implicit_capture):
                weight *= np.exp(-l*Sig_a[cell])

            #still in the slab?
            if (np.fabs(x-thickness) < 1.0e-14) or (x > thickness):
                leak_right += weight
                alive = 0
            elif (x<= 1.0e-14):
                alive = 0
                leak_left += weight
            else:
                cell= int(x/dx) #compute cell particle collision is in
                if (implicit_capture):
                    if (collide):
                        mu = random.uniform(-1,1)
                        scalar_flux[cell] += weight*iSig_s[cell]/dx
                else: #scatter or absorb
                    scalar_flux[cell] += weight*iSig_t[cell]/dx
                    if (collide) and (random.random() < Sig_s[cell]*iSig_t[cell]):
                        #scatter, pick new mu
                        mu = random.uniform(-1,1)
                    elif (collide): #absorbed
                        alive = 0

    return leak_left,leak_right, scalar_flux, scalar_flux_tl/dx, X, N

## Functions for final project

For your final project, there are a number of approaches that you might take to create your solver. Here are some helpful functions that might be useful as starting points for you in doing your project. 

In [ ]:
def create_particles(N,Q,X,Y,dx,dy):
    """Create N source particles in 2-D regular grid with source strengths in the 2-D array Q
    Inputs:
    N:         Number of neutrons to create
    Q:         2-D array of source strengths
    X,Y:       2-D array of zone centers
    dx,dy:     Width and height of zones
    
    Returns:
    census:    N by 7 array containing, weight, position (x,y), mu, gamma, and zone numbers
    """
    total = np.sum(Q)
    I,J = Q.shape
    census = np.empty((1,7))
    for i in range(I):
        for j in range(J):
            if Q[i,j] > 1.0e-14:
                num_emit = (np.ceil(Q[i,j]/total*N))
                #set weight
                wgt = Q[i,j]*dx*dy/(num_emit+1.0e-14)
                for emit in range(int(num_emit)):
                    #set position
                    pos = np.random.uniform(-0.5,0.5,2)
                    x_pos = dx * pos[0] + X[i,j]
                    y_pos = dy * pos[1] + Y[i,j]
                    mu = np.random.uniform(-1,1,1)
                    gamma = np.random.uniform(0,2*np.pi,1)
                    census = np.vstack((census,[wgt,x_pos,y_pos,mu[0],gamma[0],i,j]))
    return np.delete(census,0,axis=0)

def move_particles(census,X,Y,dx,dy,Sig_t,Sig_a,implicit_capture = True):
    """Create N source particles in 2-D regular grid with source strengths in the 2-D array Q
    Inputs:
    census:    List of particles created by the source function
    X,Y:       2-D arrays of cell centers
    dx,dy:     Widths of zones
    Sig_t:     2-D array of total macroscopic cross-sections
    Sig_a:     2-D array of absorption macroscopic cross-sections
    implicit_capture: whether or not to use implicit capture tracking
    
    Returns:
    scalar_flux_coll:    collision-estimated scalar flux array the same size as X and Y
    scalar_flux_tl:      track-length-estimated scalar flux array the same size as X and Y
    """
    Sig_s = Sig_t - Sig_a
    scalar_flux_coll = 0*X + 1e-14
    scalar_flux_tl =  0*X + 1e-14
    Lx, Ly = X.shape
    for neut in census:
        alive = 1
        while (alive):
            cell = np.array(neut[5:7], dtype=int)
            #compute distance to collision
            if (implicit_capture):
                #get distance to collision
                l = -np.log(1-np.random.random(1))/(Sig_s[cell[0], cell[1]] + 1.0e-14)
            else:
                #get distance to collision
                l = -np.log(1-np.random.random(1))/Sig_t[cell[0], cell[1]]
            #distance to x boundary
            center = [ X[cell[0], cell[1]], Y[cell[0], cell[1]]]
            pos = neut[1:3]
            mu = neut[3]
            gamma = neut[4]
            omega_x = np.sqrt(1.0-mu*mu)*np.cos(gamma)
            omega_y = np.sqrt(1.0-mu*mu)*np.sin(gamma)
            if (omega_x > 0):
                dist_x = (center[0] + dx*0.5 - pos[0])/omega_x + 1.0e-14
            else:
                dist_x = -(pos[0] - (center[0] - dx*0.5))/omega_x + 1.0e-14
            if (omega_y > 0):
                dist_y = (center[1] + dy*0.5 - pos[1])/omega_y + 1.0e-14
            else:
                dist_y = -(pos[1] - (center[1] - dy*0.5))/omega_y + 1.0e-14
            assert(dist_y>0)
            assert(dist_x>0)
            
            #find smallest distance
            if (l < dist_x) and (l < dist_y):
                neut[1] += l*omega_x
                neut[2] += l*omega_y
                #score in collision tally
                if (implicit_capture):
                    scalar_flux_coll[cell[0], cell[1]] += neut[0]/Sig_s[cell[0], cell[1]]
                else:
                    scalar_flux_coll[cell[0], cell[1]] += neut[0]/Sig_t[cell[0], cell[1]]
                if (implicit_capture and (Sig_a[cell[0], cell[1]] > 0)):
                    scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                 np.exp(-l*Sig_a[cell[0], cell[1]]))
                                                                /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                else:
                    scalar_flux_tl[cell[0],cell[1]] += neut[0]*l
                
                if (implicit_capture):
                    neut[3] = np.random.uniform(-1,1,1)
                    neut[4] = np.random.uniform(0,2*np.pi,1)
                    neut[0] *= np.exp(-l*Sig_a[cell[0], cell[1]] )
                else:
                    #scatter or absorb
                    if (np.random.random(1) < Sig_s[cell[0], cell[1]]/Sig_t[cell[0], cell[1]]): 
                        #scatter, pick new mu
                        neut[3] = np.random.uniform(-1,1,1)
                        neut[4] = np.random.uniform(0,2*np.pi,1)
                    else: #absorbed
                        #print("killed")
                        alive = 0
            elif (l >= dist_x) or (l >= dist_y):
                if (dist_y < dist_x):
                    pos[0] += (dist_y)*omega_x
                    neut[6] += np.sign(omega_y)
                    pos[1] += (dist_y + 1e-10)*omega_y
                    neut[1] = pos[0]
                    neut[2] = pos[1]
                    if (implicit_capture) and (Sig_a[cell[0], cell[1]] > 0):
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                     np.exp(-dist_y*Sig_a[cell[0], cell[1]]))
                                                                    /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                        neut[0] *= np.exp(-dist_y*Sig_a[cell[0], cell[1]] )
                    else:
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*dist_y
                else:
                    pos[1] += (dist_x)*omega_y
                    neut[5] += np.sign(omega_x)
                    pos[0] += (dist_x + 1e-10)*omega_x
                    neut[1] = pos[0]
                    neut[2] = pos[1]
                    if (implicit_capture) and (Sig_a[cell[0], cell[1]] > 0):
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                     np.exp(-dist_x*Sig_a[cell[0], cell[1]]))
                                                                    /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                        neut[0] *= np.exp(-dist_x*Sig_a[cell[0], cell[1]] )
                    else:
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*dist_x
            else:
                assert(0==1)
            
            #are we still in the problem?
            if ((pos[0] >= np.max(X)+ dx*0.5) or (pos[1] >= np.max(Y)+ dy*0.5) or 
                 ((pos[0]) < 1.0e-8) or ((pos[1]) < 1.0e-8)) :
                alive = 0
            if (neut[5] >= Lx) or (neut[5] < 0):
                alive = 0
            if (neut[6] >= Ly) or (neut[6] < 0):
                alive = 0
    return scalar_flux_coll/dx/dy, scalar_flux_tl/dx/dy

def lattice(Lengths,Dims):
    I = Dims[0]
    J = Dims[1]
    L = I*J
    Nx = Lengths[0]
    Ny = Lengths[1]
    hx,hy = np.array(Lengths)/np.array(Dims)
    
    Sigma_t = np.ones((I,J))*1
    Sigma_a = 0*Sigma_t
    Q = 0*Sigma_t
    for j in range(J):
        for i in range(I):
            x = (i+0.5)*hx
            y = (j+0.5)*hy

            if (x>=3.0) and (x<=4.0): 
                if (y>=3.0) and (y<=4.0):
                    Q[i,j] = 1.0
                if (y>=1.0) and (y<=2.0):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
            if ( ((x>=1.0) and (x<=2.0)) or ((x>=5.0) and (x<=6.0))): 
                if ( ((y>=1.0) and (y<=2.0)) or
                    ((y>=3.0) and (y<=4.0)) or
                    ((y>=5.0) and (y<=6.0))):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
            if ( ((x>=2.0) and (x<=3.0)) or ((x>=4.0) and (x<=5.0))): 
                if ( ((y>=2.0) and (y<=3.0)) or
                    ((y>=4.0) and (y<=5.0))):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
    return Sigma_t, Sigma_a, Q
        
def expfiss(x):
    return 0.453*math.exp(-1.036*x)*math.sinh(math.sqrt(2.29*x))
